# EDA · Real Food Classifier

Análisis exploratorio del dataset de Open Food Facts para predecir el **grado de procesamiento NOVA** de un producto (1: sin procesar · 2: ingredientes culinarios · 3: procesado · 4: ultraprocesado).

Este notebook cubre la **fase 1** del proyecto: limpieza común y análisis. El resultado es un CSV limpio en `data/processed/` del que partimos las tres para entrenar los modelos.

## Paso 1 · Descargar y filtrar

### 1.1 Columnas disponibles

El export completo de Open Food Facts pesa 1,4 GB comprimido. Antes de cargarlo, leemos solo la cabecera (`nrows=0`) para ver qué columnas tiene y elegir las que nos interesan.

In [1]:
import pandas as pd

RAW = "../data/raw/off_products.csv.gz"

cols = pd.read_csv(RAW, sep="\t", nrows=0).columns
print(f"Columnas totales: {len(cols)}")

interesantes = [c for c in cols if any(k in c for k in ["nova", "_100g", "pnns", "additives", "ingredients_text", "countries"])]
interesantes

Columnas totales: 211


['countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'nova_group',
 'pnns_groups_1',
 'pnns_groups_2',
 'energy-kj_100g',
 'energy-kcal_100g',
 'energy_100g',
 'energy-from-fat_100g',
 'fat_100g',
 'saturated-fat_100g',
 'butyric-acid_100g',
 'caproic-acid_100g',
 'caprylic-acid_100g',
 'capric-acid_100g',
 'lauric-acid_100g',
 'myristic-acid_100g',
 'palmitic-acid_100g',
 'stearic-acid_100g',
 'arachidic-acid_100g',
 'behenic-acid_100g',
 'lignoceric-acid_100g',
 'cerotic-acid_100g',
 'montanic-acid_100g',
 'melissic-acid_100g',
 'unsaturated-fat_100g',
 'monounsaturated-fat_100g',
 'omega-9-fat_100g',
 'polyunsaturated-fat_100g',
 'omega-3-fat_100g',
 'omega-6-fat_100g',
 'alpha-linolenic-acid_100g',
 'eicosapentaenoic-acid_100g',
 'docosahexaenoic-acid_100g',
 'linoleic-acid_100g',
 'arachidonic-acid_100g',
 'gamma-linolenic-acid_100g',
 'dihomo-gamma-linolenic-acid_100g',
 'oleic-acid_100g',
 'elai

### 1.2 Carga por trozos y filtro a España

Para no llenar la memoria:
- leemos el fichero en trozos de 200.000 filas (`chunksize`);
- cargamos solo 23 de las 211 columnas (`usecols`): identificación, categoría, ingredientes (texto y normalizados), aditivos, NOVA y nutrientes principales por 100 g;
- nos quedamos solo con productos vendidos en España (`countries_tags` contiene `en:spain`).

**No cargamos** `nutriscore_score` ni `nutriscore_grade`: el objetivo es NOVA y el Nutri-Score no debe entrar al modelo.

El resultado se guarda en `data/interim/off_spain.csv.gz` para no repetir esta lectura, que tarda varios minutos.

In [2]:
USECOLS = [
    "code", "product_name", "brands",
    "countries_tags", "categories_tags", "pnns_groups_1", "pnns_groups_2",
    "ingredients_text", "additives_n", "additives_tags",
    "nova_group",
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g",
    "carbohydrates_100g", "sugars_100g", "fiber_100g",
    "proteins_100g", "salt_100g", "sodium_100g",
    "fruits-vegetables-legumes_100g", "ingredients_tags", "ingredients_analysis_tags",
]

faltan = [c for c in USECOLS if c not in cols]
print("Columnas que no existen:", faltan)

trozos = []
total = 0
for chunk in pd.read_csv(
        RAW, sep="\t", usecols=USECOLS,
        chunksize=200_000,
        low_memory=False,
        on_bad_lines="skip",
):
    total += len(chunk)
    es = chunk[chunk["countries_tags"].str.contains("en:spain", na=False)]
    trozos.append(es)

df = pd.concat(trozos, ignore_index=True)
print(f"Filas totales: {total:,} -> España: {len(df):,}")

df.to_csv("../data/interim/off_spain.csv.gz", index=False)

Columnas que no existen: []
Filas totales: 4,906,687 -> España: 362,954


### 1.3 Embudo de filas y distribución de NOVA

Contamos cuántos productos sobreviven a cada filtro y cómo se reparten las clases.

In [3]:
print(f"Productos en España:        {len(df):,}")
print(f"Con NOVA:                   {df['nova_group'].notna().sum():,}")
print(f"Con NOVA e ingredientes:    {(df['nova_group'].notna() & df['ingredients_text'].notna()).sum():,}")

df["nova_group"].value_counts(dropna=False).sort_index()

Productos en España:        362,954
Con NOVA:                   54,403
Con NOVA e ingredientes:    46,561


nova_group
1.0      6996
2.0      8038
3.0     10390
4.0     28979
NaN    308551
Name: count, dtype: int64

**Resultados:**

| | Productos | % sobre el anterior |
|---|---|---|
| Export completo | 4.906.687 | |
| En España | 362.954 | 7,4 % |
| Con NOVA | 54.403 | 15 % |
| Con NOVA e ingredientes | 46.561 | 86 % |

| NOVA | Productos | % |
|---|---|---|
| 1 · sin procesar | 6.996 | 12,9 % |
| 2 · ingredientes culinarios | 8.038 | 14,8 % |
| 3 · procesados | 10.390 | 19,1 % |
| 4 · ultraprocesados | 28.979 | 53,3 % |

**Decisiones:**
- El 85 % de los productos españoles no tiene NOVA y no sirve para entrenar. Con ~54.000 productos etiquetados hay datos suficientes, así que **no ampliamos a otros países**.
- Desbalance moderado: la clase 4 es la mitad del dataset. **Mantenemos las 4 clases** y evaluaremos con **F1 macro** y recall por clase, no solo con accuracy.
- `nova_group` aparece como decimal (`1.0`) por los nulos. Se convertirá a entero al eliminarlos.

### 1.4 Productos con NOVA

Eliminamos los productos sin NOVA (no conocemos su clase, así que no sirven para entrenar) y pasamos `nova_group` a entero.

A partir de aquí cargamos el fichero intermedio, sin repetir la lectura del export completo.

In [4]:
df = pd.read_csv(
    "../data/interim/off_spain.csv.gz",
    low_memory=False,
    dtype={"code": str},
)

df = df[df["nova_group"].notna()].copy()

df["nova_group"] = df["nova_group"].astype(int)

print(df.shape)
df["nova_group"].value_counts().sort_index()

(54403, 23)


nova_group
1     6996
2     8038
3    10390
4    28979
Name: count, dtype: int64

## Paso 2 · Columnas que se eliminan

Al elegir `USECOLS` ya dejamos fuera las columnas con fuga de datos (Nutri-Score) y los metadatos (URLs, fechas, autor). Ahora revisamos las columnas cargadas: porcentaje de nulos y si aportan algo al modelo.

In [5]:
nulos = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
nulos

additives_tags                    47.6
fiber_100g                        39.6
fruits-vegetables-legumes_100g    17.0
ingredients_tags                  14.5
ingredients_text                  14.4
ingredients_analysis_tags         14.4
additives_n                       14.4
saturated-fat_100g                12.0
sodium_100g                       11.8
salt_100g                         11.8
sugars_100g                       11.7
proteins_100g                     11.2
fat_100g                          11.1
carbohydrates_100g                11.0
brands                            10.9
energy-kcal_100g                  10.4
categories_tags                    4.9
product_name                       2.4
code                               0.0
pnns_groups_2                      0.0
nova_group                         0.0
countries_tags                     0.0
pnns_groups_1                      0.0
dtype: float64

In [6]:
df.head()

,code,product_name,brands,categories_tags,countries_tags,ingredients_text,ingredients_tags,ingredients_analysis_tags,additives_n,additives_tags,...,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits-vegetables-legumes_100g
0,9800300037,gum cool watermelon,Tic Tac,"en:snacks,en:sweet-snacks,en:confectioneries,e...","en:france,en:spain,en:united-states","XYLITOL, GUM BASE, ISOMALT, GUM ARABIC, SORBIT...","en:e967,en:gum-base,en:e953,en:e414,en:e420,en...","en:palm-oil-free,en:vegan-status-unknown,en:ve...",12.0,"en:e129,en:e321,en:e334,en:e414,en:e420,en:e55...",...,333.0,0.0,0.0,67.0,0.0,NaN,0.0,0.000000,0.000000,0.000000
1,8422947520014,Seitán en lonchas,Soria Natural,"en:plant-based-foods-and-beverages,en:plant-ba...",en:spain,"_Seitán_ 83% (agua, gluten de _trigo_, harina ...","en:seitan,en:cereal,en:wheat,es:caldo-de-cocci...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,...,139.0,0.5,0.1,13.0,0.5,2.2,20.0,0.850000,0.340000,5.244531
2,13029,Cheddar,McDonald's,"en:dairies,en:fermented-foods,en:fermented-mil...","en:denmark,en:france,en:norway,en:poland,en:sp...","_Cheddar_ (51%), eau, _fromage_ (9%), _lait_ é...","en:cheddar,en:dairy,en:cheese,en:water,en:skim...","en:may-contain-palm-oil,en:non-vegan,en:maybe-...",5.0,"en:e160a,en:e160ai,en:e160c,en:e322,en:e322i,e...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
3,11210115644,Tabasco Rojo,McIlhenny Company,"en:condiments,en:sauces,en:hot-sauces,en:groce...","en:belgium,en:spain","Vinagre, chile rojo (19%), sal.","en:vinegar,en:red-chili-pepper,en:chili-pepper...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,...,16.0,0.7,0.2,1.6,0.0,1.3,1.0,1.800000,0.720000,19.000000
5,12000171741,Diet Pepsi,Pepsi,"en:beverages-and-beverages-preparations,en:bev...","en:spain,en:united-states","Carbonated water, caramel color, aspartame, ph...","en:carbonated-water,en:water,en:e150a,en:e951,...","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",6.0,"en:e150a,en:e212,en:e330,en:e338,en:e950,en:e951",...,0.0,0.0,NaN,0.0,NaN,NaN,0.0,0.025381,0.010152,0.000000


### 2.1 Comprobaciones antes de decidir

- ¿Los nulos de `additives_tags` son productos sin aditivos?
- ¿Faltan los nutrientes todos a la vez?
- ¿`pnns_groups` esconde valores desconocidos como texto?
- ¿Hay alguna columna de ingredientes normalizada?

In [7]:
# 1. Nulos de additives_tags
sin_tags = df["additives_tags"].isna()
print("additives_tags nulos, según additives_n:")
print(df.loc[sin_tags, "additives_n"].value_counts(dropna=False).head())

# 2. ¿Cuántos productos no tienen NINGÚN nutriente principal?
NUTRIENTES = ["energy-kcal_100g", "fat_100g", "saturated-fat_100g",
              "carbohydrates_100g", "sugars_100g", "proteins_100g", "salt_100g"]
print("\nSin ningún nutriente:", df[NUTRIENTES].isna().all(axis=1).sum())
print("Con algún nutriente pero no todos:", (df[NUTRIENTES].isna().any(axis=1) & df[NUTRIENTES].notna().any(axis=1)).sum())

# 3. Valores de pnns: "unknown" o similares
print("\npnns_groups_1:")
print(df["pnns_groups_1"].value_counts())

# 4. Columnas de ingredientes en el export original (cols es de la celda 1)
print("\nColumnas de ingredientes:", [c for c in cols if "ingredient" in c])

additives_tags nulos, según additives_n:
additives_n
0.0    18075
NaN     7842
Name: count, dtype: int64

Sin ningún nutriente: 5097
Con algún nutriente pero no todos: 2645

pnns_groups_1:
pnns_groups_1
Sugary snacks              11031
unknown                     8423
Fat and sauces              7193
Cereals and potatoes        6181
Beverages                   5221
Milk and dairy products     4865
Fish Meat Eggs              3643
Fruits and vegetables       3122
Salty snacks                2180
Composite foods             1832
Alcoholic beverages          549
Baby foods                   162
Name: count, dtype: int64

Columnas de ingredientes: ['ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'image_ingredients_url', 'image_ingredients_small_url']


### 2.2 Decisiones

- `additives_tags`: 18.075 nulos son productos **sin aditivos** (`additives_n = 0`), no datos perdidos. Se rellenan con lista vacía en el paso 5.
- 5.097 productos no tienen **ningún** nutriente: se eliminan en el paso 5. Otros 2.645 tienen nutrientes parciales: se imputan.
- `pnns_groups_1` tiene 8.423 valores `"unknown"`. Se conservan como categoría propia.
- Se ha añadido `ingredients_tags` (ingredientes normalizados en inglés) e `ingredients_analysis_tags` para crear variables en el paso 7.

**Columnas eliminadas:**
- `countries_tags`: tras filtrar por España no aporta información.
- `categories_tags`: lista de categorías muy larga y variable; `pnns_groups_1` y `pnns_groups_2` ya resumen la categoría del producto.

**Columnas que no entran al modelo pero se conservan:** `code`, `product_name` y `brands`, para detectar duplicados (paso 6) y para la app.

In [8]:
df = df.drop(columns=["countries_tags", "categories_tags"])

# Identificadores
ID_COLS = ["code", "product_name", "brands"]

print(df.shape)
df.columns.tolist()

(54403, 21)


['code',
 'product_name',
 'brands',
 'ingredients_text',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'additives_n',
 'additives_tags',
 'nova_group',
 'pnns_groups_1',
 'pnns_groups_2',
 'energy-kcal_100g',
 'fat_100g',
 'saturated-fat_100g',
 'carbohydrates_100g',
 'sugars_100g',
 'fiber_100g',
 'proteins_100g',
 'salt_100g',
 'sodium_100g',
 'fruits-vegetables-legumes_100g']

## Paso 3 · Valores sucios y unidades

Comprobamos:
- que las columnas numéricas se han leído como números;
- que `salt_100g` y `sodium_100g` dicen lo mismo (sal ≈ sodio × 2,5);
- qué valores tiene `pnns_groups_2`;
- el formato de las columnas de ingredientes normalizados.

In [9]:
df.dtypes

code                                  str
product_name                          str
brands                                str
ingredients_text                      str
ingredients_tags                      str
ingredients_analysis_tags             str
additives_n                       float64
additives_tags                        str
nova_group                          int64
pnns_groups_1                         str
pnns_groups_2                         str
energy-kcal_100g                  float64
fat_100g                          float64
saturated-fat_100g                float64
carbohydrates_100g                float64
sugars_100g                       float64
fiber_100g                        float64
proteins_100g                     float64
salt_100g                         float64
sodium_100g                       float64
fruits-vegetables-legumes_100g    float64
dtype: object

In [10]:
# Solo filas con las dos columnas y sodio > 0 (para no dividir entre cero)
mask = df["salt_100g"].notna() & df["sodium_100g"].notna() & (df["sodium_100g"] > 0)
ratio = df.loc[mask, "salt_100g"] / df.loc[mask, "sodium_100g"]

# Si es la misma información, el ratio será ~2,5 en casi todas las filas
print(ratio.describe())
print("\n% de filas con ratio entre 2,4 y 2,6:", round(ratio.between(2.4, 2.6).mean() * 100, 1))

# ¿Hay productos con sodio pero sin sal? Esos se podrían recuperar
print("Sodio sin sal:", (df["salt_100g"].isna() & df["sodium_100g"].notna()).sum())
print("Sal sin sodio:", (df["salt_100g"].notna() & df["sodium_100g"].isna()).sum())

count    3.974100e+04
mean     1.524583e+17
std      3.039279e+19
min      0.000000e+00
25%      2.500000e+00
50%      2.500000e+00
75%      2.500000e+00
max      6.058847e+21
dtype: float64

% de filas con ratio entre 2,4 y 2,6: 97.4
Sodio sin sal: 0
Sal sin sodio: 0


In [11]:
# ¿Esconde también "unknown"? ¿Hay grupos muy pequeños?
df["pnns_groups_2"].value_counts()

pnns_groups_2
unknown                             8423
Sweets                              5903
Fats                                4571
Biscuits and cakes                  3418
Dressings and sauces                2622
Cereals                             2061
Milk and yogurt                     1900
Vegetables                          1862
Cheese                              1781
Bread                               1697
Fish and seafood                    1549
Chocolate products                  1477
One-dish meals                      1255
Breakfast cereals                   1140
Artificially sweetened beverages    1139
Processed meat                      1094
Waters and flavored waters           929
Meat                                 895
Appetizers                           872
Plant-based milk substitutes         825
Teas and herbal teas and coffees     818
Sweetened beverages                  817
Legumes                              711
Ice cream                            673
Sa

In [12]:
# Listas
for col in ["ingredients_tags", "ingredients_analysis_tags", "additives_tags"]:
    print(col)
    print(df[col].dropna().iloc[0][:300], "\n")

ingredients_tags
en:e967,en:gum-base,en:e953,en:e414,en:e420,en:natural-and-artificial-flavouring,en:artificial-flavouring,en:natural-flavouring,en:flavouring,en:e553b,en:e553,en:LESS THAN 2% OF,en:e950,en:e334,en:e955,en:e903,en:e129,en:e321,en:CITRIC ACID MAGNESIUM STEARATE,en:TO MAINTAIN FREHSNESS 

ingredients_analysis_tags
en:palm-oil-free,en:vegan-status-unknown,en:vegetarian-status-unknown 

additives_tags
en:e129,en:e321,en:e334,en:e414,en:e420,en:e553,en:e553b,en:e572,en:e903,en:e950,en:e953,en:e955,en:e967 



In [13]:
# Un tag "limpio" tiene prefijo de idioma y palabras en minúscula con guiones: en:palm-oil
patron_limpio = r"^[a-z]{2}:[a-z0-9-]+$"

tags = df["ingredients_tags"].dropna().str.split(",").explode()
print(f"Tags totales: {len(tags):,}")
print(f"% limpios:    {tags.str.match(patron_limpio).mean() * 100:.1f}")

# ¿Qué idiomas aparecen? (si hay muchos es:/fr:, son ingredientes sin traducir)
print("\nPrefijos:")
print(tags.str[:3].value_counts().head(8))

Tags totales: 1,052,091
% limpios:    95.0

Prefijos:
ingredients_tags
en:    946124
es:     68877
fr:     12388
ar:      8991
pt:      4177
de:      2349
ca:      2340
it:      1540
Name: count, dtype: int64


In [14]:
df = df.drop(columns=["sodium_100g"])
print(df.shape)

(54403, 20)


### 3.1 Decisiones

- **Tipos:** correctos. No hay conversiones pendientes.
- **Sal y sodio:** en el 97,4 % de las filas sal = sodio × 2,5 y sus nulos coinciden. Se elimina `sodium_100g`. Los ratios absurdos (2,6 %) son errores de tecleo y se tratan en el paso 4.
- **`pnns_groups_2`:** 42 subcategorías, con los mismos 8.423 `unknown` que `pnns_groups_1`. Se conserva para imputar nulos (paso 5); qué nivel de categoría entra al modelo se decide en el paso 10.
- **Listas de ingredientes y aditivos:** texto separado por comas con prefijo de idioma. En `ingredients_tags`, ~90 % de los tags están reconocidos en inglés (`en:`), ~6,5 % quedan en español sin traducir (`es:`) y ~5 % son texto sin reconocer. Además, incluye los ingredientes "padre" (por ejemplo `flavouring` junto a `natural-flavouring`), así que no sirve para contar ingredientes directamente. `additives_tags` e `ingredients_analysis_tags` están limpias. Se trabajará en el paso 7.